# 002: Tokenization Systems — Character, word, and BPE subword tokenization from scratch

## 1. TOKENIZATION LEVELS
- **Character-level**: Each character is a token. Small vocabulary but very long sequences.
- **Word-level**: Each word is a token. Large vocabulary, OOV (out-of-vocabulary) problem for rare words.
- **Subword (BPE - Byte Pair Encoding)**: Iteratively merges the most frequent character pairs. Balances vocabulary size and sequence length.

## 2. BPE ALGORITHM
1. Initialize vocabulary with individual characters + end-of-word marker.
2. Count all adjacent character pairs in the corpus.
3. Merge the most frequent pair into a new token.
4. Repeat until desired vocabulary size is reached.
---


In [ ]:
from collections import Counter, defaultdict

class BPETokenizer:
    """Byte Pair Encoding tokenizer from scratch."""
    def __init__(self):
        self.merges = []
        self.vocab = set()

    def _get_pairs(self, word_freqs):
        """Count all adjacent symbol pairs across the corpus."""
        pairs = Counter()
        for word, freq in word_freqs.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    def train(self, corpus, num_merges=10):
        """Train BPE on a corpus (list of words)."""
        # Initialize: split each word into characters + end marker
        word_freqs = Counter()
        for word in corpus:
            spaced = ' '.join(list(word)) + ' </w>'
            word_freqs[spaced] += 1

        for i in range(num_merges):
            pairs = self._get_pairs(word_freqs)
            if not pairs:
                break
            # Find the most frequent pair
            best_pair = max(pairs, key=pairs.get)
            self.merges.append(best_pair)
            # Merge the best pair in all words
            new_word_freqs = {}
            bigram = ' '.join(best_pair)
            replacement = ''.join(best_pair)
            for word, freq in word_freqs.items():
                new_word = word.replace(bigram, replacement)
                new_word_freqs[new_word] = freq
            word_freqs = new_word_freqs
            print(f"Merge {i+1}: {best_pair} → '{replacement}' (freq={pairs[best_pair]})")

        # Build final vocabulary
        for word in word_freqs:
            for token in word.split():
                self.vocab.add(token)

    def tokenize(self, word):
        """Apply learned merges to a new word."""
        symbols = list(word) + ['</w>']
        for (a, b) in self.merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == a and symbols[i+1] == b:
                    symbols = symbols[:i] + [a + b] + symbols[i+2:]
                else:
                    i += 1
        return symbols



In [ ]:
corpus = ["low"] * 5 + ["lower"] * 2 + ["newest"] * 6 + ["widest"] * 3
print("--- Training BPE Tokenizer ---")
tokenizer = BPETokenizer()
tokenizer.train(corpus, num_merges=10)

print(f"\nVocabulary ({len(tokenizer.vocab)} tokens): {sorted(tokenizer.vocab)}")
print(f"\nTokenize 'lowest': {tokenizer.tokenize('lowest')}")
print(f"Tokenize 'newer':  {tokenizer.tokenize('newer')}")
